# Index Directional — Signal Analysis

**Four signals on two instruments (CAC-TR 1× and LVC 2×):**

| # | Signal | Logic |
|---|--------|-------|
| 1 | **B\&H** | Always long (benchmark) |
| 2 | **IVol Z-score** | Long when CAC IVol rolling 126d z-score < 1.0 |
| 3 | **TKAN v3** | Long when model predicts net-positive 5d return |
| 4 | **TKAN + IVol** | *Both* conditions must hold (regime-gated TKAN) |

Run cells in order. Cells 2–3 load from sfera_db and the TKAN weights cache — no retraining needed.

In [ ]:
# ── Cell 1: Imports & config ──────────────────────────────────────────────────
import os, sys, json, pickle, pathlib, warnings
warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

import numpy as np
import pandas as pd
from scipy import stats as scipy_stats

# workspace root on sys.path so sfera_db and signum are importable
try:
    _nb_dir = pathlib.Path(__vscode_ipynb_path__).resolve().parent
except NameError:
    _nb_dir = pathlib.Path(os.getcwd())

_WORKSPACE = _nb_dir.parents[3]           # btest/research/Index Directional → workspace root
_BTEST     = _nb_dir.parents[1]           # btest/
_TKAN_V3   = _nb_dir / 'tkan' / 'v3'
_WEIGHTS   = _TKAN_V3 / 'weights'
_OUTPUT    = _BTEST / 'outputs' / 'idx_directional'
_OUTPUT.mkdir(parents=True, exist_ok=True)

for _p in [str(_WORKSPACE), str(_WORKSPACE / 'sfera-db')]:
    if _p not in sys.path: sys.path.insert(0, _p)

# ── model config (must match training notebook) ────────────────────────────────
WINDOW_SIZE      = 30
PREDICTION_DAYS  = 5
TARGET_TYPE      = 'path5d_dm'
BACKTEST_START   = '2015-01-01'
IVOL_EXIT_WINDOW = 126
TKAN_THR         = 0.0     # long when predicted 5d cum-return >= this
IVOL_Z_THR       = 1.0     # long when IVol z-score < this

FEATURE_COLS = [
    'log_return_1d', 'high_low_range', 'close_to_high',
    'close_vs_sma15',
    'ivol_zscore', 'ivol_ema_ratio', 'ivol_pctl', 'ivol_roc5',
    'rvol_park20_zscore', 'vol_spread',
    'return_5d', 'return_20d',
]

print(f'Weights  : {_WEIGHTS}')
print(f'Output   : {_OUTPUT}')
print(f'TKAN thr : {TKAN_THR}   |   IVol Z thr : {IVOL_Z_THR}')

In [ ]:
# ── Cell 2: Load data from sfera_db ──────────────────────────────────────────
import sfera_db

cactr = sfera_db.query(
    "SELECT trade_date AS date, close_price AS close "
    "FROM bbgidx.index_total_return WHERE ticker = 'CACT' ORDER BY trade_date"
).assign(date=lambda d: pd.to_datetime(d['date'])).set_index('date')

cac_ohlc = sfera_db.query(
    "SELECT trade_date AS date, open_price AS open, high_price AS high, "
    "low_price AS low, close_price AS cac_close "
    "FROM bbgidx.index_prices WHERE ticker = 'CAC' ORDER BY trade_date"
).assign(date=lambda d: pd.to_datetime(d['date'])).set_index('date')

ivol_raw = sfera_db.query(
    'SELECT trade_date AS date, "3m_50d_ivol" AS ivol '
    "FROM bbgidx.index_implied_vol WHERE ticker = 'CAC' ORDER BY trade_date"
).assign(date=lambda d: pd.to_datetime(d['date'])).set_index('date')[['ivol']]

common = cactr.index.intersection(cac_ohlc.index).intersection(ivol_raw.index)
df = cac_ohlc.loc[common].copy()
df['close'] = cactr.loc[common, 'close']
df['ivol']  = ivol_raw.loc[common, 'ivol']

# ── features (same as training) ───────────────────────────────────────────────
df['log_return_1d']  = np.log(df['close'] / df['close'].shift(1))
df['high_low_range'] = (df['high'] - df['low']) / df['close'].shift(1).replace(0, np.nan)
df['close_to_high']  = (df['high'] - df['cac_close']) / (df['high'] - df['low'] + 1e-9)
df['sma15']          = df['close'].rolling(15).mean()
df['close_vs_sma15'] = df['close'] / df['sma15'] - 1
df['return_5d']      = np.log(df['close'] / df['close'].shift(5))
df['return_20d']     = np.log(df['close'] / df['close'].shift(20))
hl_sq = np.log(df['high'] / df['low'].replace(0, np.nan)) ** 2
park  = np.sqrt((1 / (4 * np.log(2))) * hl_sq.rolling(20).mean() * 252)
close_rvol = df['log_return_1d'].rolling(20).std() * np.sqrt(252)
df['rvol_park20']        = park.where(park.notna() & (park > 0), close_rvol)
ivol = df['ivol']
df['ivol_ewma20']        = ivol.ewm(span=20).mean()
df['ivol_zscore']        = (ivol - ivol.rolling(IVOL_EXIT_WINDOW).mean()) / \
                            (ivol.rolling(IVOL_EXIT_WINDOW).std() + 1e-9)
df['ivol_ema_ratio']     = ivol / (df['ivol_ewma20'] + 1e-9)
df['ivol_pctl']          = ivol.rolling(IVOL_EXIT_WINDOW).apply(
                               lambda x: pd.Series(x).rank(pct=True).iloc[-1], raw=False)
df['ivol_roc5']          = ivol.pct_change(5)
df['rvol_park20_zscore'] = (df['rvol_park20'] - df['rvol_park20'].rolling(IVOL_EXIT_WINDOW).mean()) / \
                            (df['rvol_park20'].rolling(IVOL_EXIT_WINDOW).std() + 1e-9)
df['vol_spread']         = ivol - df['rvol_park20']

# ── LVC 2× ───────────────────────────────────────────────────────────────────
try:
    lvc_raw = sfera_db.query(
        "SELECT trade_date AS date, close_price AS close "
        "FROM instruments.etf_prices WHERE ticker = 'LVC' ORDER BY trade_date"
    ).assign(date=lambda d: pd.to_datetime(d['date'])).set_index('date')
    has_lvc = True
except Exception:
    _csv = _BTEST / 'data' / 'lvc_ohlcv.csv'
    if _csv.exists():
        lvc_raw = pd.read_csv(_csv, parse_dates=['date']).set_index('date')[['close']]
        has_lvc = True
    else:
        lvc_raw = None; has_lvc = False

bt_start = pd.Timestamp(BACKTEST_START)
df_bt = df.loc[df.index >= bt_start].copy()

print(f'CAC data : {df_bt.index[0].date()} → {df_bt.index[-1].date()}  ({len(df_bt):,} rows)')
print(f'LVC data : {"found" if has_lvc else "NOT found — will run CAC-TR only"}')
df_bt[['close', 'ivol', 'ivol_zscore']].tail(3)

In [ ]:
# ── Cell 3: Load TKAN predictions (from cache — no retraining) ───────────────
import hashlib

def _fingerprint():
    sig = f'{WINDOW_SIZE}|{PREDICTION_DAYS}|{TARGET_TYPE}|{sorted(FEATURE_COLS)}'
    return hashlib.md5(sig.encode()).hexdigest()[:12]

cache_path = _WEIGHTS / 'pred_cache.pkl'
current_fp = _fingerprint()
pred_df = None

if cache_path.exists():
    with open(cache_path, 'rb') as f:
        _cached = pickle.load(f)
    if (isinstance(_cached, tuple) and len(_cached) == 3
            and isinstance(_cached[0], pd.DataFrame)
            and _cached[2] == current_fp):
        pred_df, retrain_dates, _ = _cached
        print(f'✅  TKAN cache loaded: {len(pred_df):,} predictions'
              f'  {pred_df.index[0].date()} → {pred_df.index[-1].date()}')
    else:
        print('⚠   Cache fingerprint mismatch — run the TKAN_v3_research notebook to retrain')
else:
    print('⚠   No pred_cache.pkl found. Run TKAN_v3_research.ipynb cells 4–6 first.')

if pred_df is not None:
    pred_cum = pred_df.sum(axis=1).reindex(df_bt.index).fillna(0)
    print(f'\nPredicted 5d return — μ={pred_cum.mean():.4f}  σ={pred_cum.std():.4f}')
    print(f'Bull days (≥ {TKAN_THR}): {(pred_cum >= TKAN_THR).sum():,} / {len(pred_cum):,}'
          f'  ({(pred_cum >= TKAN_THR).mean()*100:.1f}%)')
else:
    pred_cum = pd.Series(0, index=df_bt.index)  # fallback: no TKAN signal

In [ ]:
# ── Cell 4: Build the 4 signals & compute metrics ────────────────────────────
# Execution: signal at close[T] → position opens at close[T+1]  (1-day lag)

ivol_z   = df_bt['ivol_zscore']
sig_ivol = (ivol_z < IVOL_Z_THR).astype(float)
sig_tkan = (pred_cum >= TKAN_THR).astype(float)

SIGNALS = {
    'B&H':          pd.Series(1.0, index=df_bt.index),
    'IVol Z<1.0':   sig_ivol,
    'TKAN v3':      sig_tkan,
    'TKAN + IVol':  (sig_tkan * sig_ivol).astype(float),
}

def metrics(strat_ret, bench_ret, pos, label):
    sr = strat_ret.fillna(0)
    br = bench_ret.reindex(sr.index).fillna(0)
    n_y  = len(sr) / 252
    ann  = sr.mean() * 252
    vol  = sr.std() * 252**0.5
    sh   = ann / vol if vol > 0 else 0
    eq   = (1 + sr).cumprod()
    mdd  = float((eq / eq.cummax() - 1).min()) * 100
    cagr = (float(eq.iloc[-1]) ** (1 / n_y) - 1) * 100 if n_y > 0 else 0
    calmar = cagr / abs(mdd) if mdd != 0 else float('nan')
    down = sr[sr < 0]
    sortino = ann / (down.std() * 252**0.5) if len(down) > 1 else 0
    if br.std() > 0:
        beta  = float(np.cov(sr.values, br.values)[0, 1] / np.var(br.values))
        alpha = (ann - beta * br.mean() * 252) * 100
    else:
        beta = alpha = float('nan')
    p = pos.reindex(sr.index).fillna(0)
    active = sr[p > 0]
    win = float((active > 0).mean() * 100) if len(active) > 0 else float('nan')
    return {
        'Signal':      label,
        'CAGR %':      round(cagr, 2),
        'Sharpe':      round(sh, 3),
        'Sortino':     round(sortino, 3),
        'Max DD %':    round(mdd, 2),
        'Calmar':      round(calmar, 3) if not (isinstance(calmar, float) and calmar != calmar) else '—',
        'Beta':        round(beta,  3)  if not (isinstance(beta,  float) and beta  != beta)  else '—',
        'Alpha %/yr':  round(alpha, 2)  if not (isinstance(alpha, float) and alpha != alpha) else '—',
        'Win %':       round(win, 1)    if not (isinstance(win,   float) and win   != win)   else '—',
        'In-Mkt %':    round(float(pos.reindex(sr.index).fillna(0).mean() * 100), 1),
        'Total %':     round((float(eq.iloc[-1]) - 1) * 100, 1),
    }

bench_ret = np.log(df_bt['close'] / df_bt['close'].shift(1))

INSTRUMENTS = {'CAC-TR (1×)': bench_ret}
if has_lvc:
    lvc_common = lvc_raw.index.intersection(df_bt.index)
    lvc_close  = lvc_raw.loc[lvc_common, 'close']
    lvc_ret    = np.log(lvc_close / lvc_close.shift(1)).reindex(df_bt.index)
    INSTRUMENTS['LVC (2×)'] = lvc_ret

# ── store equity curves for charting ─────────────────────────────────────────
equity_curves = {}
all_rows = {}

for instr_name, daily_ret in INSTRUMENTS.items():
    rows = []
    equity_curves[instr_name] = {}
    for sig_name, raw_sig in SIGNALS.items():
        pos_sig  = raw_sig.reindex(daily_ret.index).fillna(0)
        pos      = pos_sig.shift(1).fillna(0)
        strat_r  = daily_ret * pos
        bench_r  = daily_ret
        equity_curves[instr_name][sig_name] = (1 + strat_r.fillna(0)).cumprod()
        rows.append(metrics(strat_r, bench_r, pos, sig_name))
    all_rows[instr_name] = rows

# ── print tables ─────────────────────────────────────────────────────────────
def _print_table(rows, title):
    cols = list(rows[0].keys())
    w    = {c: max(len(c), max(len(str(r.get(c,''))) for r in rows)) + 2 for c in cols}
    sep  = '+' + '+'.join('-'*w[c] for c in cols) + '+'
    print(f'\n  {title}')
    print(sep)
    print('|' + '|'.join(f' {c:<{w[c]-1}}' for c in cols) + '|')
    print(sep)
    for r in rows:
        print('|' + '|'.join(f' {str(r.get(c,"")):<{w[c]-1}}' for c in cols) + '|')
    print(sep)

for instr_name, rows in all_rows.items():
    _print_table(rows, f'Instrument: {instr_name}  |  {BACKTEST_START} → {df_bt.index[-1].date()}')

# save CSV
combined = [{'Instrument': k, **r} for k, rows in all_rows.items() for r in rows]
out_csv = _OUTPUT / 'metrics.csv'
pd.DataFrame(combined).to_csv(out_csv, index=False)
print(f'\n💾  Saved → {out_csv}')

In [ ]:
# ── Cell 5: Signum chart ──────────────────────────────────────────────────────
# Price pane with entry shading, predicted signal pane, equity comparison
from signum import Chart, Dashboard

CHART_INSTR = 'CAC-TR (1×)'    # change to 'LVC (2×)' if you want the leveraged view
CHART_SIG   = 'TKAN + IVol'    # change to any key in SIGNALS

daily_ret = INSTRUMENTS[CHART_INSTR]
raw_sig   = SIGNALS[CHART_SIG]
pos       = raw_sig.reindex(daily_ret.index).fillna(0).shift(1).fillna(0)

def _ts(s): return pd.DataFrame({'time': s.index.strftime('%Y-%m-%d'), 'value': s.values})

price_idx  = (1 + daily_ret.fillna(0)).cumprod()
shade_pos  = pd.DataFrame({'time': df_bt.index.strftime('%Y-%m-%d'), 'position': pos.values})

# Pane 1 — price + entry shading
p1 = Chart(height=380, watermark='CAC40 TR')
p1.line(_ts(price_idx), name='CAC40 TR', color='#4169E1', width=2)
p1.shade(shade_pos, color='#22c55e', opacity=0.15)

# Pane 2 — TKAN predicted return OR IVol z-score
if pred_df is not None:
    signal_series = pred_cum
    p2_label = f'TKAN predicted 5d return  (thr={TKAN_THR})'
else:
    signal_series = df_bt['ivol_zscore']
    p2_label = f'IVol Z-score  (thr={IVOL_Z_THR})'
p2 = Chart(height=140, watermark=p2_label).baseline(_ts(signal_series), base_value=0)

# Pane 3 — equity curves for all 4 signals
COLOURS = {'B&H': '#64748b', 'IVol Z<1.0': '#f59e0b', 'TKAN v3': '#818cf8', 'TKAN + IVol': '#22c55e'}
p3 = Chart(height=200, watermark=f'Equity — {CHART_INSTR}')
for sig_name, eq in equity_curves[CHART_INSTR].items():
    w = 2 if sig_name == CHART_SIG else 1
    p3.line(_ts(eq), name=sig_name, color=COLOURS.get(sig_name, '#fff'), width=w)

Dashboard(
    panes=[p1, p2, p3],
    titles=[
        f'{CHART_INSTR} — {CHART_SIG} entries (green shading)',
        p2_label,
        'Equity curves — all 4 signals',
    ],
    theme='dark',
).show()

In [ ]:
# ── Cell 6: Threshold sweep — find the best TKAN threshold ──────────────────
# Optional: sweep TKAN_THR from p5 to p95 of the pred_cum distribution
# and show how Sharpe / Calmar change.  Skip if no TKAN predictions.

if pred_df is None:
    print('No TKAN predictions — skipping sweep')
else:
    thresholds = sorted(set(
        round(v, 4)
        for v in np.percentile(pred_cum.dropna(), np.linspace(5, 75, 20))
    ))

    sweep_rows = []
    daily_ret  = INSTRUMENTS['CAC-TR (1×)']
    ivol_mask  = sig_ivol.reindex(daily_ret.index).fillna(0)

    for thr in thresholds:
        for use_ivol in [False, True]:
            raw = (pred_cum >= thr).astype(float)
            if use_ivol:
                raw = (raw * ivol_mask).astype(float)
            pos       = raw.reindex(daily_ret.index).fillna(0).shift(1).fillna(0)
            strat_r   = daily_ret * pos
            ann       = strat_r.mean() * 252
            vol_s     = strat_r.std() * 252**0.5
            sh        = ann / vol_s if vol_s > 0 else 0
            eq        = (1 + strat_r.fillna(0)).cumprod()
            mdd       = float((eq / eq.cummax() - 1).min()) * 100
            cagr      = (float(eq.iloc[-1]) ** (252 / len(eq)) - 1) * 100
            calmar    = cagr / abs(mdd) if mdd != 0 else float('nan')
            sweep_rows.append({
                'TKAN_thr':  thr,
                'IVol gate': 'yes' if use_ivol else 'no',
                'In-Mkt %': round(pos.mean()*100, 1),
                'Sharpe':   round(sh, 3),
                'Calmar':   round(calmar, 3) if calmar == calmar else float('nan'),
                'CAGR %':   round(cagr, 2),
                'MaxDD %':  round(mdd, 2),
            })

    sweep_df = pd.DataFrame(sweep_rows)
    print('\n── Threshold sweep on CAC-TR ────────────────────────────────────────────────')
    print(sweep_df.sort_values('Sharpe', ascending=False).to_string(index=False))

    best = sweep_df.sort_values('Sharpe', ascending=False).iloc[0]
    print(f"\n🏆  Best combo: TKAN_thr={best['TKAN_thr']}  IVol gate={best['IVol gate']}  "
          f"Sharpe={best['Sharpe']}  Calmar={best['Calmar']}")

In [ ]:
# ── Cell 7: Signal ratio analysis ───────────────────────────────────────────
# "Total ratio" = each signal's Sharpe / B&H Sharpe
# Also shows how much of B&H drawdown is recovered and CAGR lift

if 'all_rows' not in dir():
    print('⚠  Run Cell 4 first (signals + metrics)')
else:
    for instr_name, rows in all_rows.items():
        bh = next(r for r in rows if r['Signal'] == 'B&H')
        bh_sharpe = bh['Sharpe']
        bh_cagr   = bh['CAGR %']
        bh_mdd    = abs(bh['Max DD %'])

        ratio_rows = []
        for r in rows:
            sh_ratio   = round(r['Sharpe']  / bh_sharpe, 2) if bh_sharpe != 0 else '—'
            cagr_lift  = round(r['CAGR %']  - bh_cagr, 2)
            mdd_recov  = round(1 - abs(r['Max DD %']) / bh_mdd, 3) if bh_mdd != 0 else '—'
            ratio_rows.append({
                'Signal':        r['Signal'],
                'Sharpe':        r['Sharpe'],
                'Sharpe / B&H':  sh_ratio,
                'CAGR %':        r['CAGR %'],
                'CAGR lift pp':  cagr_lift,
                'Max DD %':      r['Max DD %'],
                'DD recovered':  mdd_recov,
                'In-Mkt %':      r['In-Mkt %'],
            })

        _print_table(ratio_rows, f'Ratio analysis — {instr_name}')

    # Best combined score: Sharpe/B&H * DD_recovered (penalises staying-in-cash too aggressively)
    print('\n── Combined score = (Sharpe/B&H) × (1 + DD_recovered) ─────────────────────')
    for instr_name, rows in all_rows.items():
        bh = next(r for r in rows if r['Signal'] == 'B&H')
        bh_sh = bh['Sharpe'] ; bh_mdd = abs(bh['Max DD %'])
        scored = []
        for r in rows:
            sh_r = r['Sharpe'] / bh_sh if bh_sh else 0
            mdd_r = 1 - abs(r['Max DD %']) / bh_mdd if bh_mdd else 0
            score = round(sh_r * (1 + mdd_r), 3)
            scored.append((score, r['Signal']))
        scored.sort(reverse=True)
        print(f'  {instr_name}:')
        for sc, nm in scored:
            print(f'    {nm:<22} score={sc}')

In [ ]:
# ── Cell 8: Signal variant audit ────────────────────────────────────────────
# Lists every signal currently defined and flags redundancy.
# "Redundant" = in-market % within 2pp of another signal (nearly identical position series)

if 'SIGNALS' not in dir():
    print('⚠  Run Cell 4 first')
else:
    print('── Defined signal variants ──────────────────────────────────────────────────')
    for name, sig in SIGNALS.items():
        pct = sig.mean() * 100
        transitions = (sig.diff().abs() > 0).sum()
        print(f'  {name:<24}  in-mkt={pct:5.1f}%  transitions={transitions}')

    # Overlap matrix: % of days where two signals agree (both in or both out)
    print('\n── Pairwise overlap (% days in same position) ───────────────────────────────')
    names = list(SIGNALS.keys())
    header = f"{'':22}" + ''.join(f'{n[:10]:>12}' for n in names)
    print(header)
    overlap_data = {}
    for a in names:
        row = f'{a[:22]:<22}'
        for b in names:
            agree = (SIGNALS[a] == SIGNALS[b]).mean() * 100
            overlap_data[(a, b)] = agree
            row += f'{agree:11.1f}%'
        print(row)

    # Flag redundant pairs (>95% overlap, excluding self)
    print('\n── Redundant pairs (>95% overlap) ──────────────────────────────────────────')
    found = False
    for i, a in enumerate(names):
        for b in names[i+1:]:
            ov = overlap_data[(a, b)]
            if ov > 95:
                print(f'  ⚠  {a}  ↔  {b}  ({ov:.1f}% identical)')
                found = True
    if not found:
        print('  ✅  No redundant pairs found (all pairs differ by > 5% of days)')

In [ ]:
# ── Cell 9: Segmented mini-tests (pass/fail per signal section) ─────────────
# Run each signal logic independently with sanity checks.
# Helps catch broken signals before running the full backtest.

import traceback

PASS = '✅ PASS'; FAIL = '❌ FAIL'

def _run_test(name, fn):
    try:
        result = fn()
        status = PASS
        detail = result if isinstance(result, str) else 'ok'
    except Exception as e:
        status = FAIL
        detail = str(e)
    print(f'  {status}  {name:<40}  {detail}')

print('── Segmented signal tests ───────────────────────────────────────────────────')

# T1: sfera_db query returns non-empty data
_run_test('sfera_db CAC data loaded', lambda:
    f'{len(df_bt):,} rows  {df_bt.index[0].date()}→{df_bt.index[-1].date()}'
    if 'df_bt' in dir() and len(df_bt) > 100 else (_ for _ in ()).throw(ValueError('df_bt missing or empty'))
)

# T2: IVol z-score defined and bounded
_run_test('IVol z-score computed', lambda:
    f'range [{df_bt["ivol_zscore"].min():.2f}, {df_bt["ivol_zscore"].max():.2f}]'
    if df_bt['ivol_zscore'].notna().mean() > 0.9 else (_ for _ in ()).throw(ValueError('>10% NaN'))
)

# T3: TKAN pred_cache loaded (or gracefully absent)
_run_test('TKAN predictions available', lambda:
    f'{len(pred_df):,} rows' if pred_df is not None
    else 'no cache — signals will be IVol-only (expected if not trained)'
)

# T4: All 4 signals are binary 0/1
def _check_binary():
    bad = [n for n, s in SIGNALS.items() if not set(s.dropna().unique()).issubset({0.0, 1.0})]
    if bad: raise ValueError(f'Non-binary signals: {bad}')
    return 'all signals are 0/1'
_run_test('All signals are binary 0/1', _check_binary)

# T5: No signal is always-on or always-off (trivial)
def _check_non_trivial():
    trivial = [n for n, s in SIGNALS.items() if s.mean() in (0.0, 1.0) and n != 'B&H']
    if trivial: raise ValueError(f'Trivial signals: {trivial}')
    return 'no trivial signals'
_run_test('No trivial (always-on/off) signals', _check_non_trivial)

# T6: TKAN + IVol combined is subset of TKAN alone
def _check_subset():
    if 'TKAN v3' not in SIGNALS or 'TKAN + IVol' not in SIGNALS:
        return 'skipped (TKAN not available)'
    a = SIGNALS['TKAN v3'] ; b = SIGNALS['TKAN + IVol']
    extra = (b > a).sum()
    if extra > 0: raise ValueError(f'Combined fires {extra} days when TKAN alone does not')
    pct = b.mean() / a.mean() * 100 if a.mean() > 0 else 0
    return f'combined={pct:.1f}% of TKAN-alone days'
_run_test('TKAN+IVol ⊆ TKAN-alone', _check_subset)

# T7: Equity curves are monotone-increasing on B&H (no negative compound)
def _check_equity():
    bh_eq = equity_curves['CAC-TR (1×)']['B&H']
    min_v = bh_eq.min()
    if min_v <= 0: raise ValueError(f'Non-positive equity value: {min_v:.4f}')
    return f'B&H equity min={min_v:.3f}'
_run_test('B&H equity always > 0', _check_equity)

# T8: Metrics CSV was written
def _check_csv():
    if not out_csv.exists(): raise FileNotFoundError(str(out_csv))
    n = len(pd.read_csv(out_csv))
    return f'{n} rows in {out_csv.name}'
_run_test('Metrics CSV written', _check_csv)

print('\n── Summary ─────────────────────────────────────────────────────────────────')